# Phase 2: T4 GPU Pipeline Test

Tests the full adversarial decode pipeline on a T4 GPU to verify:
1. Timing fits within the CI budget (~20 min for inflate)
2. Score matches local evaluation
3. No OOM or compatibility issues

**Runtime:** Select GPU runtime: Runtime > Change runtime type > T4 GPU

In [ ]:
# Verify T4 GPU is available
!nvidia-smi

In [ ]:
# Clone the repo (includes model weights via LFS)
!git lfs install
!git clone https://github.com/commaai/comma_video_compression_challenge.git /content/repo
%cd /content/repo
!git lfs pull

In [ ]:
# Install Python dependencies (Colab already has torch with CUDA)
!pip install -q einops timm safetensors segmentation-models-pytorch tqdm pillow av

# Install nvidia-dali for GPU-accelerated video loading (same as CI)
!pip install -q --extra-index-url https://pypi.nvidia.com nvidia-dali-cuda120

# Add __future__ annotations for Python 3.8 compat (if needed)
!grep -q "from __future__" modules.py || sed -i '1s/^/from __future__ import annotations\n/' modules.py

import torch
print(f"torch {torch.__version__}, CUDA {torch.version.cuda}")
try:
    import nvidia.dali
    print(f"DALI {nvidia.dali.__version__}")
except ImportError:
    print("DALI not available, evaluate will use CPU")

In [ ]:
# Verify model weights and video exist
import os
for f in ['models/segnet.safetensors', 'models/posenet.safetensors', 'videos/0.mkv']:
    sz = os.path.getsize(f)
    print(f'{f}: {sz:,} bytes ({sz/1e6:.1f} MB)')
    assert sz > 1_000_000, f'{f} seems too small -- LFS pull may have failed'

In [ ]:
# Verify torch + CUDA setup
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'Compute capability: {torch.cuda.get_device_capability()}')

## Upload Submission Files

Upload the following files from your local `submissions/phase2/` directory:
- `__init__.py`
- `encode.py`
- `inflate.py`
- `inflate.sh`
- `compress.sh`

In [ ]:
# Create submission directory and upload files
!mkdir -p /content/repo/submissions/phase2

from google.colab import files
print('Upload all files from submissions/phase2/ (encode.py, inflate.py, inflate.sh, compress.sh, __init__.py):')
uploaded = files.upload()
for name, data in uploaded.items():
    path = f'/content/repo/submissions/phase2/{name}'
    with open(path, 'wb') as f:
        f.write(data)
    print(f'  Saved {path} ({len(data):,} bytes)')

## Step 1: Encode (extract targets from video)

In [ ]:
%%time
# Run encode.py to extract SegNet/PoseNet targets
!python -m submissions.phase2.encode videos/0.mkv submissions/phase2/archive/0

In [ ]:
# Create archive.zip (simulating compress.sh)
import shutil, os
archive_dir = 'submissions/phase2/archive'
zip_path = 'submissions/phase2/archive.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)

# zip the archive directory contents
!cd submissions/phase2/archive && zip -r ../archive.zip .
print(f'Archive size: {os.path.getsize(zip_path):,} bytes')

## Step 2: Inflate (adversarial decode on T4) -- THE CRITICAL TIMING TEST

Key parameters in `inflate.py` to tune for T4:
- `TARGET_ITERS=180` -- max iterations per batch
- `MIN_ITERS=60` -- minimum iterations (floor for adaptive allocation)
- `WALL_BUDGET=1400` -- seconds allocated to inflate (with compile)
- `torch.compile(mode='reduce-overhead')` -- 2x per-iter speedup via CUDA Graphs (Linux only)
- Compile warmup (3 dummy forward/backward passes) pays JIT cost upfront (~90s vs ~250s during batch 1)
- Last batch padded to batch_size=16 to avoid recompilation
- PoseNet skipped in first 30% of iterations (saves ~12% compute per iter, tested on T4)

The adaptive system automatically adjusts per-batch iterations based on:
1. **Wall clock budget** -- ensures total time fits in WALL_BUDGET
2. **Batch difficulty** -- harder batches (more class boundaries) get ~1.4x more iters

In [ ]:
%%time
# Clear any previous output
!rm -f submissions/phase2/inflated/0.raw

# Run inflate.py -- this is the step that must fit in ~20 min on T4
!python -u -m submissions.phase2.inflate submissions/phase2/archive/0 submissions/phase2/inflated/0.raw

In [ ]:
# Parse inflate output to analyze per-batch timing and difficulty
import re

# Manual paste area - paste inflate output between triple quotes if needed:
inflate_log = """
PASTE INFLATE OUTPUT HERE
"""

pattern = r'\[(\d+)/(\d+)\].*?\| ([\d.]+)s \((\d+)it, d=([\d.]+)\) \| seg=([\d.]+) pose=([\d.]+) \| mm=(\d+)/(\d+)'
matches = re.findall(pattern, inflate_log)
if not matches:
    print("No batch log data found. Paste inflate output in the cell above.")
else:
    total_time = sum(float(m[2]) for m in matches)
    total_iters = sum(int(m[3]) for m in matches)
    avg_tpi = total_time / total_iters if total_iters > 0 else 0

    print(f"Parsed {len(matches)} batches:")
    print(f"  Total optimize time: {total_time:.1f}s ({total_time/60:.1f} min)")
    print(f"  Avg iters/batch: {total_iters/len(matches):.0f}")
    print(f"  Avg time/iter: {avg_tpi*1000:.1f}ms")
    print(f"  Projected @ 150 iters: {avg_tpi*150*len(matches):.0f}s ({avg_tpi*150*len(matches)/60:.1f} min)")

    diffs = [float(m[4]) for m in matches]
    segs = [float(m[5]) for m in matches]
    print(f"\n  Difficulty: {min(diffs):.2f} - {max(diffs):.2f}")
    print(f"  Iters: {min(int(m[3]) for m in matches)} - {max(int(m[3]) for m in matches)}")
    print(f"  Seg loss: {min(segs):.4f} - {max(segs):.4f}")

    overhead_est = 180  # model load + eval
    print(f"\n  Budget: {total_time:.0f}s inflate + {overhead_est}s overhead = {total_time+overhead_est:.0f}s")
    print(f"  {'PASS' if total_time + overhead_est < 1800 else 'FAIL'} (30 min = 1800s)")

In [ ]:
# Verify output
import os
raw_path = 'submissions/phase2/inflated/0.raw'
expected_size = 1200 * 874 * 1164 * 3  # 3,661,113,600
actual_size = os.path.getsize(raw_path)
print(f'Output: {actual_size:,} bytes')
print(f'Expected: {expected_size:,} bytes')
assert actual_size == expected_size, f'Size mismatch! Got {actual_size}, expected {expected_size}'

## Step 3: Evaluate (compute score)

In [ ]:
%%time
# Uses DALI+CUDA if available (matches CI), falls back to CPU
import importlib
_dali_ok = importlib.util.find_spec("nvidia.dali") is not None
_eval_device = "cuda" if _dali_ok else "cpu"
print(f"Evaluate device: {_eval_device} ({'DALI' if _dali_ok else 'AVVideoDataset'})")
!python -u evaluate.py \
    --submission-dir submissions/phase2 \
    --uncompressed-dir videos \
    --report submissions/phase2/report.txt \
    --device {_eval_device}

In [ ]:
# Display final report
with open('submissions/phase2/report.txt', 'r') as f:
    print(f.read())

## Full CI Simulation (one cell)

Run everything below in one shot: encode + zip + unzip + inflate + evaluate.
This simulates the actual CI pipeline timing as closely as possible on Colab.

In [ ]:
import time, os, importlib

print('=' * 60)
print('  FULL CI PIPELINE SIMULATION')
print('=' * 60)

t_ci = time.time()

# ── 1. Encode ──
print('\n[1/5] ENCODE (extract targets from video)...')
t = time.time()
!rm -rf submissions/phase2/archive/0 && mkdir -p submissions/phase2/archive/0
!python -u -m submissions.phase2.encode videos/0.mkv submissions/phase2/archive/0
encode_time = time.time() - t
print(f'      Encode: {encode_time:.1f}s')

# ── 2. Zip ──
print('\n[2/5] ZIP archive...')
t = time.time()
!rm -f submissions/phase2/archive.zip
!cd submissions/phase2/archive && zip -r ../archive.zip . > /dev/null
zip_time = time.time() - t
archive_size = os.path.getsize('submissions/phase2/archive.zip')
print(f'      Zip: {zip_time:.1f}s, archive: {archive_size:,} bytes ({archive_size/1024:.1f} KB)')

# ── 3. Unzip (simulates CI unzip step) ──
print('\n[3/5] UNZIP archive...')
t = time.time()
!rm -rf submissions/phase2/archive_ci && mkdir -p submissions/phase2/archive_ci
!unzip -o submissions/phase2/archive.zip -d submissions/phase2/archive_ci > /dev/null
unzip_time = time.time() - t
print(f'      Unzip: {unzip_time:.1f}s')

# ── 4. Inflate (THE CRITICAL STEP) ──
print('\n[4/5] INFLATE (adversarial decode on T4)...')
t = time.time()
!rm -f submissions/phase2/inflated/0.raw
!python -u -m submissions.phase2.inflate submissions/phase2/archive_ci/0 submissions/phase2/inflated/0.raw
inflate_time = time.time() - t
print(f'      Inflate: {inflate_time:.1f}s ({inflate_time/60:.1f} min)')

# ── 5. Evaluate ──
_dali_ok = importlib.util.find_spec("nvidia.dali") is not None
_eval_device = "cuda" if _dali_ok else "cpu"
print(f'\n[5/5] EVALUATE (device={_eval_device}, {"DALI" if _dali_ok else "CPU fallback"})...')
t = time.time()
!python -u evaluate.py \
    --submission-dir submissions/phase2 \
    --uncompressed-dir videos \
    --report submissions/phase2/report.txt \
    --device {_eval_device}
eval_time = time.time() - t
print(f'      Evaluate: {eval_time:.1f}s ({eval_time/60:.1f} min)')

# ── Summary ──
total_time = time.time() - t_ci
ci_setup_est = 300  # apt-get, uv sync, lfs pull on Azure

print(f'\n{"=" * 60}')
print(f'  TIMING SUMMARY')
print(f'{"=" * 60}')
print(f'  Encode:    {encode_time:6.1f}s  (offline, not in CI budget)')
print(f'  Zip:       {zip_time:6.1f}s  (offline, not in CI budget)')
print(f'  Unzip:     {unzip_time:6.1f}s')
print(f'  Inflate:   {inflate_time:6.1f}s  ({inflate_time/60:.1f} min)')
print(f'  Evaluate:  {eval_time:6.1f}s  ({eval_time/60:.1f} min)')
print(f'  ─────────────────────────')
print(f'  Colab total:  {total_time:.1f}s ({total_time/60:.1f} min)')

ci_time = unzip_time + inflate_time + eval_time + ci_setup_est
print(f'\n  CI estimate: {unzip_time:.0f} + {inflate_time:.0f} + {eval_time:.0f} + ~{ci_setup_est} setup = {ci_time:.0f}s ({ci_time/60:.1f} min)')
print(f'  CI budget:   1800s (30.0 min)')
if ci_time < 1800:
    margin = 1800 - ci_time
    print(f'\n  >>> PASS: {margin:.0f}s ({margin/60:.1f} min) margin <<<')
else:
    over = ci_time - 1800
    print(f'\n  >>> FAIL: Over budget by {over:.0f}s ({over/60:.1f} min) <<<')

print(f'\n{"=" * 60}')
print(f'  SCORE')
print(f'{"=" * 60}')
with open('submissions/phase2/report.txt', 'r') as f:
    print(f.read())

# Cleanup
!rm -f submissions/phase2/inflated/0.raw
!rm -rf submissions/phase2/archive_ci
print('Cleaned up large files.')

In [ ]:
# Cleanup large files to free Colab disk space
!rm -f submissions/phase2/inflated/0.raw
!rm -rf submissions/phase2/archive_test
print('Cleaned up large files')